# DVC — Data & Model Versioning

In [ ]:
# Day 33 — DVC (Data Version Control)
# In Day 32 (MLflow) we tracked EXPERIMENTS — params, metrics, models
# But what about the DATA? What if your training data changes?
#
# The problem:
#   Git tracks CODE perfectly (small text files)
#   But data files (CSV, images, model .pkl) are HUGE — Git can't handle them
#   - 500MB dataset? Git slows down
#   - 2GB model file? Git chokes
#   - Need to go back to last month's data version? Impossible without DVC
#
# Analogy:
#   Git = tracks the RECIPE (code)    → small, text-based, easy
#   DVC = tracks the INGREDIENTS (data) → large, binary, needs special handling
#   Together: you can reproduce ANY meal from ANY point in time!
#
# DVC works ALONGSIDE Git (not replacing it):
#   Git stores: code + .dvc metadata files (tiny pointers)
#   DVC stores: actual data files (in remote storage like Azure Blob, S3)
#
# Today we'll cover:
#   1. What DVC solves (Git's data problem)
#   2. dvc init & dvc add
#   3. .dvc files (how versioning works)
#   4. Remote storage setup (local, Azure Blob, S3)
#   5. dvc push / dvc pull
#   6. Switching between data versions
#   7. DVC Pipelines (dvc.yaml, dvc repro)
#
# Install: pip install dvc
# For Azure: pip install dvc[azure]
# For S3: pip install dvc[s3]

### Why DVC? The Problem with Large Files in Git

In [ ]:
# Imagine this scenario:
#
# Your ML project has:
#   code/train.py           →  5 KB   (Git handles this perfectly)
#   data/server_metrics.csv →  500 MB (Git will struggle)
#   models/best_model.pkl   →  2 GB  (Git will break)
#
# Without DVC:
#   - Push 500MB CSV to Git → repo becomes huge and slow
#   - Team member clones repo → downloads 500MB every time
#   - Data changes weekly → repo grows to 10GB+ in a few months
#   - Can't go back to last month's data version
#
# With DVC:
#   - Data stored in cheap cloud storage (Azure Blob, S3, GCS)
#   - Git only stores a tiny .dvc file (pointer to the data)
#   - Team members pull data ONLY when they need it (dvc pull)
#   - Full version history: switch to any data version anytime
#   - Repo stays fast and clean
#
# How it works:
#   git add data/server_metrics.csv     ← NO! Don't do this
#   dvc add data/server_metrics.csv     ← YES! DVC handles it
#     → creates data/server_metrics.csv.dvc (tiny pointer file)
#     → adds data/server_metrics.csv to .gitignore
#   git add data/server_metrics.csv.dvc  ← Git tracks the pointer
#   dvc push                             ← Uploads data to remote storage

print('DVC = Git for data and models')
print('Git tracks: code + .dvc files (pointers)')
print('DVC tracks: actual data files (in cloud storage)')

### DVC Init & Add

In [ ]:
# DVC commands run from the TERMINAL (like Git)
# In Jupyter, we use ! to run shell commands
#
# Step 1: Initialize DVC in a Git repo
#   !dvc init
#   This creates a .dvc/ directory (like .git/ for DVC)
#
# Step 2: Track a data file
#   !dvc add data/server_metrics.csv
#   This creates:
#     data/server_metrics.csv.dvc  (pointer file — goes into Git)
#     Adds data/server_metrics.csv to .gitignore

# Let's create a sample dataset to demonstrate

import pandas as pd
import numpy as np
import os

# Create data directory
os.makedirs('dvc_demo/data', exist_ok=True)

# Create a sample dataset (v1 — 100 servers)
np.random.seed(42)
n = 100
data_v1 = pd.DataFrame({
    'server_id': [f'srv-{i:03d}' for i in range(n)],
    'cpu': np.random.randint(10, 100, n),
    'memory': np.random.randint(15, 95, n),
    'disk': np.random.randint(10, 90, n),
    'error_count': np.random.randint(0, 50, n),
    'status': np.random.choice(['healthy', 'warning', 'critical'], n, p=[0.6, 0.3, 0.1])
})

data_v1.to_csv('dvc_demo/data/server_metrics.csv', index=False)
print(f'Dataset v1 created: {len(data_v1)} servers')
print(data_v1.head())

In [ ]:
# In a REAL project, the terminal commands would be:
#
# $ cd dvc_demo
# $ git init
# $ dvc init
#   → creates .dvc/ directory
#
# $ dvc add data/server_metrics.csv
#   → creates data/server_metrics.csv.dvc
#   → adds data/server_metrics.csv to data/.gitignore
#
# $ git add data/server_metrics.csv.dvc data/.gitignore
# $ git commit -m "Track server metrics data v1"
#
# That's it! Data is now version-controlled.

# Let's look at what a .dvc file contains

# Simulating a .dvc file content
import hashlib

# DVC uses MD5 hash to identify file versions
with open('dvc_demo/data/server_metrics.csv', 'rb') as f:
    file_hash = hashlib.md5(f.read()).hexdigest()

dvc_file_content = f"""outs:
- md5: {file_hash}
  size: {os.path.getsize('dvc_demo/data/server_metrics.csv')}
  path: server_metrics.csv
"""

print('.dvc file content (what Git stores):')
print(dvc_file_content)
print(f'Actual file size: {os.path.getsize("dvc_demo/data/server_metrics.csv")} bytes')
print(f'.dvc file would be: ~{len(dvc_file_content)} bytes (tiny!)')
print(f'\nMD5 hash is the version fingerprint — same data = same hash')

### Remote Storage

In [ ]:
# Remote storage is WHERE DVC keeps the actual data files
# (Just like Git remote is where Git pushes code)
#
# DVC supports many remote storage backends:
#   - Local folder (for learning/testing)
#   - Azure Blob Storage (our primary cloud)
#   - AWS S3
#   - Google Cloud Storage
#   - SSH/SFTP servers
#   - HDFS (Hadoop)
#
# Setup commands:
#
# Local storage (for learning):
#   $ dvc remote add -d mylocal /tmp/dvc-storage
#
# Azure Blob Storage (production):
#   $ dvc remote add -d azureblob azure://my-container/dvc-storage
#   $ dvc remote modify azureblob account_name mystorageaccount
#
# AWS S3:
#   $ dvc remote add -d s3remote s3://my-bucket/dvc-storage
#
# The -d flag means "default" — all push/pull commands use this remote

# Let's simulate a local remote storage

os.makedirs('dvc_demo/local_remote', exist_ok=True)

print('DVC Remote Storage Setup Commands:')
print()
print('# For learning (local folder):')
print('dvc remote add -d localstorage ./local_remote')
print()
print('# For Azure Blob Storage (production):')
print('dvc remote add -d azblob azure://mldata-container/dvc-store')
print('dvc remote modify azblob account_name myazurestorageacct')
print()
print('# Push data to remote:')
print('dvc push')
print()
print('# Pull data from remote (on a new machine):')
print('dvc pull')

### Versioning Data — The Full Workflow

In [ ]:
# The real power of DVC: VERSION your data just like code
#
# Workflow:
#   Version 1: Train with 100 servers
#   Version 2: Get 200 more servers → retrain
#   Version 3: Clean bad data → retrain
#   Boss: "v2 was actually better, go back to that data"
#   You: "No problem!" (git checkout + dvc checkout)

# Let's simulate data versioning

# v1: 100 servers (already created above)
v1_data = pd.read_csv('dvc_demo/data/server_metrics.csv')
print(f'v1: {len(v1_data)} servers')

# In a real project:
# $ dvc add data/server_metrics.csv
# $ git add data/server_metrics.csv.dvc
# $ git commit -m "Data v1: 100 servers"
# $ git tag data-v1
# $ dvc push

In [ ]:
# v2: Add 200 more servers (simulate data growth)

np.random.seed(99)
n_new = 200
new_data = pd.DataFrame({
    'server_id': [f'srv-{i:03d}' for i in range(100, 300)],
    'cpu': np.random.randint(10, 100, n_new),
    'memory': np.random.randint(15, 95, n_new),
    'disk': np.random.randint(10, 90, n_new),
    'error_count': np.random.randint(0, 50, n_new),
    'status': np.random.choice(['healthy', 'warning', 'critical'], n_new, p=[0.6, 0.3, 0.1])
})

v2_data = pd.concat([v1_data, new_data], ignore_index=True)
v2_data.to_csv('dvc_demo/data/server_metrics.csv', index=False)
print(f'v2: {len(v2_data)} servers (added {n_new} new)')

# In a real project:
# $ dvc add data/server_metrics.csv      ← DVC detects the change!
# $ git add data/server_metrics.csv.dvc
# $ git commit -m "Data v2: 300 servers (added 200 new)"
# $ git tag data-v2
# $ dvc push

print()
print('To switch between versions:')
print('  git checkout data-v1             # switch .dvc pointer')
print('  dvc checkout                     # download that version of data')
print('  # Now you have v1 data (100 servers) again!')

In [ ]:
# Let's demonstrate the hash-based versioning

import hashlib

# Hash of v1 data
v1_bytes = v1_data.to_csv(index=False).encode()
v1_hash = hashlib.md5(v1_bytes).hexdigest()

# Hash of v2 data
v2_bytes = v2_data.to_csv(index=False).encode()
v2_hash = hashlib.md5(v2_bytes).hexdigest()

print('DVC Version Tracking:')
print(f'  v1 hash: {v1_hash} ({len(v1_data)} rows)')
print(f'  v2 hash: {v2_hash} ({len(v2_data)} rows)')
print(f'  Same data? {v1_hash == v2_hash}')    # False — different data!
print()
print('DVC stores files by their hash:')
print(f'  .dvc/cache/{v1_hash[:2]}/{v1_hash[2:]}  ← v1 data')
print(f'  .dvc/cache/{v2_hash[:2]}/{v2_hash[2:]}  ← v2 data')
print()
print('This means: if you add the SAME file twice, DVC only stores it ONCE!')
print('(Same hash = same content = no duplicate storage)')

### DVC Pipelines

In [ ]:
# DVC Pipelines define your ML workflow as a YAML file (dvc.yaml)
# This makes your experiments REPRODUCIBLE
#
# Pipeline = sequence of STAGES:
#   prepare → train → evaluate
#
# Each stage defines:
#   cmd:   the command to run
#   deps:  input files (dependencies)
#   outs:  output files (artifacts)
#   metrics: metric files (for comparison)
#
# DVC only re-runs a stage if its dependencies CHANGED
# (Like a Makefile for ML!)

import yaml

# Example dvc.yaml pipeline
pipeline = {
    'stages': {
        'prepare': {
            'cmd': 'python src/prepare.py',
            'deps': ['data/raw/server_metrics.csv', 'src/prepare.py'],
            'outs': ['data/processed/train.csv', 'data/processed/test.csv']
        },
        'train': {
            'cmd': 'python src/train.py',
            'deps': ['data/processed/train.csv', 'src/train.py'],
            'outs': ['models/model.pkl'],
            'params': ['train.n_estimators', 'train.max_depth']
        },
        'evaluate': {
            'cmd': 'python src/evaluate.py',
            'deps': ['models/model.pkl', 'data/processed/test.csv', 'src/evaluate.py'],
            'metrics': [{'results/metrics.json': {'cache': False}}]
        }
    }
}

# Write it
with open('dvc_demo/dvc.yaml', 'w') as f:
    yaml.dump(pipeline, f, default_flow_style=False, sort_keys=False)

print('dvc.yaml pipeline:')
with open('dvc_demo/dvc.yaml', 'r') as f:
    print(f.read())

In [ ]:
# params.yaml — pipeline parameters
# This file stores your hyperparameters SEPARATELY from code
# So you can change params without editing Python files

import yaml

params = {
    'prepare': {
        'test_size': 0.2,
        'random_state': 42
    },
    'train': {
        'model_type': 'RandomForest',
        'n_estimators': 200,
        'max_depth': 10,
        'random_state': 42
    },
    'evaluate': {
        'threshold': 0.85
    }
}

with open('dvc_demo/params.yaml', 'w') as f:
    yaml.dump(params, f, default_flow_style=False)

print('params.yaml:')
with open('dvc_demo/params.yaml', 'r') as f:
    print(f.read())

print('Commands to run the pipeline:')
print('  dvc repro           # run all stages (only re-runs what changed)')
print('  dvc repro train     # run only the train stage + dependencies')
print('  dvc dag             # show the pipeline DAG (dependency graph)')
print('  dvc metrics show    # show metrics from the last run')
print('  dvc metrics diff    # compare metrics between Git commits')

### DVC + Git Workflow (Real-World Example)

In [ ]:
# Here's the COMPLETE workflow for a real ML project:
#
# INITIAL SETUP:
#   $ git init my-ml-project
#   $ cd my-ml-project
#   $ dvc init
#   $ dvc remote add -d azblob azure://mycontainer/dvc-store
#   $ git add .
#   $ git commit -m "Initialize DVC"
#
# ADD DATA:
#   $ dvc add data/server_metrics.csv
#   $ git add data/server_metrics.csv.dvc data/.gitignore
#   $ git commit -m "Add server metrics data v1"
#   $ dvc push              # upload data to Azure Blob
#   $ git push              # push code + .dvc file to GitHub
#
# TEAMMATE CLONES:
#   $ git clone <repo-url>
#   $ cd my-ml-project
#   $ dvc pull              # download data from Azure Blob
#   # Now they have the exact same data you have!
#
# DATA UPDATE:
#   (add new servers to CSV)
#   $ dvc add data/server_metrics.csv    # DVC detects the change
#   $ git add data/server_metrics.csv.dvc
#   $ git commit -m "Data v2: added 200 new servers"
#   $ dvc push && git push
#
# GO BACK TO OLD DATA:
#   $ git log --oneline     # find the commit hash for v1
#   $ git checkout <hash> -- data/server_metrics.csv.dvc
#   $ dvc checkout           # restore v1 data from cache
#   # Now you're working with v1 data again!

print('DVC + Git Workflow Summary:')
print()
print('  Code changes  →  git add → git commit → git push')
print('  Data changes  →  dvc add → git commit → dvc push + git push')
print('  Get data      →  git clone → dvc pull')
print('  Switch version →  git checkout → dvc checkout')
print()
print('KEY POINT: .dvc files are the bridge between Git and DVC')
print('Git sees .dvc files (tiny), DVC manages actual data (huge)')

### DVC vs MLflow — When to Use Which

In [ ]:
# DVC and MLflow complement each other — they solve DIFFERENT problems
#
# DVC (Day 33):
#   ✅ Version LARGE FILES (data, models)
#   ✅ Store data in cheap cloud storage
#   ✅ Reproducible pipelines (dvc.yaml)
#   ✅ Works with Git (extends it)
#   ❌ Does NOT have experiment UI or metric comparison
#
# MLflow (Day 32):
#   ✅ Track EXPERIMENTS (params, metrics)
#   ✅ Compare runs (UI dashboard)
#   ✅ Model registry with stages
#   ✅ Framework-agnostic model format
#   ❌ Does NOT version large datasets
#
# Together:
#   DVC versions your DATA (what you train on)
#   MLflow tracks your EXPERIMENTS (how you train)
#
# Production MLOps pipeline:
#   1. DVC pulls the right data version
#   2. Training script logs to MLflow
#   3. Best model registered in MLflow Registry
#   4. DevOps deploys from MLflow Registry (FastAPI, Day 34)
#   5. DVC pipeline ensures reproducibility

comparison = {
    'Feature':         ['Data versioning', 'Experiment tracking', 'Model registry', 'Pipeline DAG', 'Cloud storage', 'Web UI'],
    'DVC':             ['✅ Core feature',  '❌ Not built-in',     '❌ No',           '✅ dvc.yaml',   '✅ push/pull',  '❌ CLI only'],
    'MLflow':          ['❌ Not built-in',  '✅ Core feature',     '✅ Yes',           '❌ No',         '✅ Artifacts',  '✅ Web dashboard']
}

import pandas as pd
df = pd.DataFrame(comparison)
print(df.to_string(index=False))

In [ ]:
# Let's simulate a complete DVC-tracked ML project structure

import os
import json

# Create a realistic project structure
project_dir = 'dvc_demo/ml_project'
dirs = [
    f'{project_dir}/data/raw',
    f'{project_dir}/data/processed',
    f'{project_dir}/src',
    f'{project_dir}/models',
    f'{project_dir}/results'
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# Create a prepare.py script
prepare_code = '''import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

# Load params
with open('params.yaml') as f:
    params = yaml.safe_load(f)

# Load raw data
df = pd.read_csv('data/raw/server_metrics.csv')

# Split
train, test = train_test_split(df, test_size=params['prepare']['test_size'],
                                random_state=params['prepare']['random_state'])

# Save processed
train.to_csv('data/processed/train.csv', index=False)
test.to_csv('data/processed/test.csv', index=False)
print(f'Train: {len(train)}, Test: {len(test)}')
'''

with open(f'{project_dir}/src/prepare.py', 'w') as f:
    f.write(prepare_code)

# Show the project structure
print('ML Project Structure with DVC:')
print()
for root, dirs_list, files in os.walk(project_dir):
    level = root.replace(project_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    folder = os.path.basename(root)
    print(f'{indent}{folder}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

print()
print('Files tracked by Git: .py, .yaml, .dvc, .gitignore')
print('Files tracked by DVC: .csv, .pkl (via .dvc pointer files)')

In [ ]:
# Practice Questions:

# 1. Create a dataset of 500 API response times
#    Version it as v1 (save hash)
#    Add 200 more records → save as v2 (save hash)
#    Verify the hashes are different

# 2. Write a params.yaml for a classification pipeline with:
#    - prepare: test_size, random_state
#    - train: model_type, n_estimators, max_depth, learning_rate
#    - evaluate: accuracy_threshold, f1_threshold
#    Load it and print each section

# 3. Create a dvc.yaml pipeline with 4 stages:
#    download → prepare → train → evaluate
#    Define deps and outs for each stage
#    Save it and read it back

# 4. Write a function version_data(filepath) that:
#    - Calculates MD5 hash of the file
#    - Creates a .dvc-like metadata dict (hash, size, path)
#    - Returns the metadata
#    Test with different versions of a CSV file

### DevOps / AI-MLOps Practice

In [ ]:
# DVC brings DevOps practices (versioning, CI/CD, reproducibility) to DATA

# Exercise 1: Data Versioning Manager
# Create a class DataVersionManager that simulates DVC operations:
#   __init__(self, project_path)
#   add(self, filepath) → calculate hash, create .dvc metadata, track it
#   versions(self, filepath) → list all versions (hash, size, timestamp)
#   checkout(self, filepath, version_hash) → restore a specific version
#   status(self) → show all tracked files and their current versions
# Test: add a CSV, modify it, add again, list versions, checkout v1

# Exercise 2: CI/CD Pipeline Config Generator
# Write a function generate_dvc_pipeline(stages_config) that:
#   Takes a list of stage configs:
#     [{'name': 'prepare', 'script': 'prepare.py', 'deps': [...], 'outs': [...]}, ...]
#   Generates a valid dvc.yaml file
#   Generates a params.yaml with default hyperparameters
#   Returns both as dicts
# Test with a 3-stage ML pipeline

# Exercise 3: Reproducibility Checker
# Write a function check_reproducibility(project_path) that:
#   - Reads dvc.yaml to get all stages
#   - For each stage, checks if deps files exist
#   - For each stage, checks if outs files exist
#   - Reports: which stages need re-running (deps changed but outs are stale)
#   - Returns a status dict: {'prepare': 'up-to-date', 'train': 'needs-rerun'}

# Why this matters for me as a DevOps/MLOps engineer:
# 1. DVC integrates into CI/CD pipelines (GitHub Actions, Azure DevOps)
# 2. CI pipeline: git push → dvc pull → dvc repro → dvc push → deploy
# 3. Every experiment is reproducible — same data + same code = same result
# 4. Data stored in Azure Blob is cheaper than Git LFS
# 5. Team collaboration: everyone works with the same data version

# Clean up demo files
import shutil
if os.path.exists('dvc_demo'):
    shutil.rmtree('dvc_demo')
    print('Demo files cleaned up!')